# 第 0 天：真实数据准备 —— 从模拟到实战的桥梁

> 所属阶段：课程前置准备
> 今日主题：搭建真实A股数据管线
> 必做：获取行情+财务数据
> 选做：对接行业分类与指数数据
> 目标产出：一份可直接用于后续29天课程的 `(dates × assets)` 面板数据

---

## 0. 为什么需要今天？

本课程第1-30天的所有notebook都使用 `np.random` 生成的模拟数据。这让你可以快速理解方法论，但也带来一个关键gap：

> 模拟数据没有停牌、没有复权、没有财务披露滞后、没有股票池变化、没有真实的市场结构。

当你学完全部30天方法论后，面对真实A股数据时会发现：**80%的时间花在数据清洗上，而不是因子研究上。**

今天的任务就是提前搭好这座桥，让你后续29天的学习可以直接在真实数据上运行。

---

## 1. 你需要准备的数据

### 1.1 日频行情数据（必备）

| 字段 | 说明 | 示例 |
|------|------|------|
| `trade_date` | 交易日 | 2024-01-02 |
| `ts_code` | 股票代码 | 000001.SZ |
| `open` | 开盘价（后复权） | 12.50 |
| `high` | 最高价（后复权） | 12.80 |
| `low` | 最低价（后复权） | 12.30 |
| `close` | 收盘价（后复权） | 12.60 |
| `pre_close` | 前收盘价（后复权） | 12.40 |
| `volume` | 成交量（股） | 50000000 |
| `amount` | 成交额（元） | 630000000 |

> **关键：使用后复权价格。** 前复权会改变历史价格，导致回测失真。不复权会遇到除权除息跳空。

### 1.2 财务数据（必备）

| 字段 | 说明 | 频率 |
|------|------|------|
| `ts_code` | 股票代码 | — |
| `end_date` | 报告期 | 季/年 |
| `ann_date` | 公告日期 | — |
| `total_assets` | 总资产 | 季 |
| `total_equity` | 股东权益 | 季 |
| `net_profit` | 净利润 | 季 |
| `operating_revenue` | 营业收入 | 季 |
| `operating_cash_flow` | 经营性现金流 | 季 |
| `roe` | ROE | 季 |
| `roa` | ROA | 季 |
| `gross_profit_margin` | 毛利率 | 季 |
| `pe` / `pb` / `ps` | 估值指标 | 日 |

> **关键：注意披露滞后。** Q1数据要到4月底才全部披露，在4月之前只能用上一年的年报。

### 1.3 辅助数据（推荐）

| 数据 | 用途 |
|------|------|
| 申万行业分类 | 行业中性化 |
| 沪深300/中证500成分股 | 股票池筛选 |
| ST/退市标记 | 股票池过滤 |
| 无风险利率（SHIBOR/国债） | Sharpe比率计算 |

---

## 2. 数据获取方案

### 方案A：AKShare（免费，推荐入门）


In [ ]:
pip install akshare


In [ ]:
import akshare as ak
import pandas as pd
from datetime import datetime

# --- 获取A股日频行情（后复权） ---
# 单只股票示例
def get_daily_hfq(ts_code: str, start_date: str, end_date: str):
    """获取单只A股后复权日频行情"""
    df = ak.stock_zh_a_hist(
        symbol=ts_code.split(".")[0],
        period="daily",
        start_date=start_date.replace("-", ""),
        end_date=end_date.replace("-", ""),
        adjust="hfq"  # 后复权
    )
    df["ts_code"] = ts_code
    df.rename(columns={
        "日期": "trade_date",
        "开盘": "open",
        "收盘": "close",
        "最高": "high",
        "最低": "low",
        "成交量": "volume",
        "成交额": "amount",
    }, inplace=True)
    return df[["trade_date", "ts_code", "open", "high", "low", "close", "volume", "amount"]]

# --- 获取财务数据 ---
def get_financial_data(ts_code: str):
    """获取A股财务指标"""
    df = ak.stock_financial_abstract_ths(symbol=ts_code.split(".")[0])
    return df

# --- 获取行业分类 ---
def get_industry_classification():
    """获取申万行业分类"""
    df = ak.stock_info_sz_area_code()
    return df


### 方案B：Tushare（需要积分，数据更全）


In [ ]:
pip install tushare


In [ ]:
import tushare as ts

# 需要先在 https://tushare.pro 注册获取token
pro = ts.pro_api("your_token")

# 日线行情（后复权）
df = pro.daily(ts_code="000001.SZ", start_date="20200101", end_date="20241231")
# adj_factor 用于后复权计算

# 财务数据
df_fin = pro.fina_indicator(ts_code="000001.SZ", start_date="20200101")


### 方案C：JoinQuant / RiceQuant 本地数据（推荐进阶）

如果已有JoinQuant或RiceQuant账号，可以直接下载本地数据包，数据质量最高。

---

## 3. 核心数据处理流程

### 3.1 构建统一面板


In [ ]:
import numpy as np
import pandas as pd

def build_price_panel(raw_prices: pd.DataFrame) -> pd.DataFrame:
    """
    输入：长格式行情表 (trade_date, ts_code, close, volume, ...)
    输出：宽格式面板 (dates × assets)
    """
    price_cols = ["open", "high", "low", "close", "volume", "amount"]
    panels = {}
    for col in price_cols:
        if col in raw_prices.columns:
            panels[col] = raw_prices.pivot(
                index="trade_date", columns="ts_code", values=col
            )
    return panels

# 使用
# panels = build_price_panel(raw_df)
# close_panel = panels["close"]  # (T × N) DataFrame


### 3.2 处理停牌


In [ ]:
def handle_suspension(price_panel: pd.DataFrame, method: str = "ffill", max_gap: int = 20):
    """
    处理停牌导致的价格缺失。
    method='ffill': 前向填充（适合短期停牌）
    method='drop': 直接剔除（适合长期停牌）
    max_gap: 前向填充的最大连续缺失天数
    """
    if method == "ffill":
        # 前向填充，但限制最大连续缺失天数
        result = price_panel.copy()
        for col in result.columns:
            mask = result[col].isna()
            # 标记连续缺失超过max_gap的块
            gap_groups = (mask != mask.shift()).cumsum()
            gap_lengths = mask.groupby(gap_groups).transform("sum")
            result[col] = result[col].ffill(limit=max_gap)
            # 超过max_gap的连续缺失恢复为NaN
            result.loc[(gap_lengths > max_gap) & mask, col] = np.nan
        return result
    elif method == "drop":
        return price_panel.dropna(axis=0, how="all")
    return price_panel

# 标记ST股票（简单规则：代码以ST开头）
def filter_normal_stocks(ts_codes: list) -> list:
    """过滤掉ST和*ST股票"""
    return [c for c in ts_codes if not ("ST" in c or "*ST" in c)]


### 3.3 处理财务数据滞后


In [ ]:
def align_financial_with_lag(
    fin_data: pd.DataFrame,
    trade_dates: pd.DatetimeIndex,
    lag_months: int = 4
) -> pd.DataFrame:
    """
    将财务数据对齐到交易日，正确处理披露滞后。

    规则：Q1报告在4月底前不可用，Q2(中报)在8月底前不可用，
          Q3在10月底前不可用，年报在次年4月底前不可用。

    lag_months=4 是保守处理：假设报告期后4个月数据才可用。
    """
    fin_data = fin_data.copy()
    # 报告期 -> 可用日期 = 报告期结束 + lag_months
    fin_data["available_date"] = pd.to_datetime(fin_data["end_date"]) + pd.DateOffset(months=lag_months)

    # 将财务数据向前填充到每个交易日
    aligned = pd.DataFrame(index=trade_dates)
    for col in fin_data.columns:
        if col not in ["ts_code", "end_date", "ann_date", "available_date"]:
            # 对每个交易日，取最近可用的财务数据
            pass  # 实际实现略复杂，建议使用 merge_asof
    return aligned


### 3.4 收益率计算


In [ ]:
def compute_returns(close_panel: pd.DataFrame) -> dict:
    """从收盘价面板计算各持有期收益率标签"""
    returns = close_panel.pct_change()  # 日收益率

    # 未来N日收益率（用于因子检验标签）
    labels = {}
    for horizon in [1, 5, 10, 20, 60]:
        labels[f"future_{horizon}d"] = close_panel.shift(-horizon) / close_panel - 1

    return {
        "daily_ret": returns,
        "future_1d": labels["future_1d"],
        "future_5d": labels["future_5d"],
        "future_20d": labels["future_20d"],
        "future_60d": labels["future_60d"],
    }


---

## 4. 数据质量检查清单

在开始因子研究前，逐项确认：

- [ ] 价格使用**后复权**，无除权除息跳空
- [ ] 停牌日有明确处理策略（前向填充或剔除）
- [ ] ST/退市股已标记或过滤
- [ ] 股票池在样本期内保持一致或记录了变更时点
- [ ] 财务数据考虑了披露滞后（至少3-4个月）
- [ ] 缺失值比例已检查（每只股票缺失超过30%则考虑剔除）
- [ ] 无未来函数：标签使用 `shift(-N)` 而非 `shift(N)`
- [ ] 涨跌停数据已标记（可选，进阶需要）

---

## 5. 你的今日任务

1. **选择数据源**：AKShare（免费）/ Tushare（积分）/ JoinQuant本地
2. **获取2019-2024年A股数据**（至少包含沪深300成分股，约300只）
3. **运行上述数据处理流程**，输出：
   - `close_panel.csv` → (T × N) 后复权收盘价
   - `volume_panel.csv` → (T × N) 成交量
   - `labels_panel.csv` → 未来1/5/20/60日收益率
   - `industry_map.csv` → 股票代码到行业的映射
4. **将输出文件保存到 `data/` 目录**，后续课程引用

完成今天的数据准备后，从明天（第1天）开始，把所有notebook中的 `np.random` 模拟数据替换为你自己的真实数据面板。
